In [1]:
import pandas as pd

In [2]:
# Notebook 위치를 기준으로 상대경로를 사용해 원본 데이터를 불러옵니다.
df = pd.read_csv("../data/raw/Car details v3.csv")

In [3]:
# 전처리 전 데이터의 행과 열 개수를 확인합니다.
df.shape

(8128, 13)

In [4]:
# 전처리 전 완전히 동일한 중복 행의 개수를 확인합니다.
df.duplicated().sum()

np.int64(1202)

In [5]:
# 원본 df는 유지하고, 각 중복 그룹의 첫 번째 행만 남긴 별도 DataFrame을 만듭니다.
df_clean = df.drop_duplicates().copy()

In [6]:
# 중복 제거 후 데이터의 행과 열 개수를 확인합니다.
df_clean.shape

(6926, 13)

In [7]:
# 중복 제거로 실제 삭제된 행의 개수를 계산합니다.
len(df) - len(df_clean)

1202

In [8]:
# 중복 제거 후 완전히 동일한 중복 행이 남아 있는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [9]:
# 중복 제거 후 각 컬럼의 결측치 개수를 확인합니다.
df_clean.isna().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          208
engine           208
max_power        205
torque           209
seats            208
dtype: int64

In [10]:
# 결측치가 있는 5개 컬럼을 지정합니다.
missing_columns = [
    "mileage",
    "engine",
    "max_power",
    "torque",
    "seats",
]

# 각 행에서 결측인 컬럼 수와 그 분포를 확인합니다.
missing_count_per_row_clean = df_clean[missing_columns].isna().sum(axis=1)

missing_count_per_row_clean.value_counts().sort_index()

0    6717
1       1
4       3
5     205
Name: count, dtype: int64

In [11]:
# 5개 컬럼이 모두 동시에 결측인 행 수를 확인합니다.
df_clean[missing_columns].isna().all(axis=1).sum()

np.int64(205)

In [12]:
# 5개 중 정확히 4개 컬럼이 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 4]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
575,Maruti Alto K10 LXI,2011,204999,97500,Petrol,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN
1442,Maruti Swift Dzire VDI Optional,2017,589000,41232,Diesel,Dealer,Manual,First Owner,NaN,NaN,0,NaN,NaN
2549,Tata Indica Vista Quadrajet LS,2012,240000,70000,Diesel,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN


In [13]:
# 5개 중 정확히 1개 컬럼만 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 1]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
4933,Maruti Omni CNG,2000,80000,100000,CNG,Individual,Manual,Second Owner,10.9 km/kg,796 CC,bhp,NaN,8.0


In [14]:
# 5개 컬럼 중 하나라도 결측인 전체 행 수를 확인합니다.
df_clean[missing_columns].isna().any(axis=1).sum()

np.int64(209)

In [15]:
# mileage의 숫자 부분과 단위 표기를 확인합니다.
mileage_text = df_clean["mileage"].dropna().astype(str).str.strip()

mileage_parts = mileage_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

mileage_parts["unit"].value_counts(dropna=False)

unit
kmpl     6631
km/kg      87
Name: count, dtype: int64

In [16]:
# mileage의 숫자 부분 추출 실패 행 수를 확인합니다.
mileage_failure_mask = mileage_parts["number"].isna()
mileage_failure_count = mileage_failure_mask.sum()

mileage_failure_count

np.int64(0)

In [17]:
# mileage의 숫자 부분 추출 실패 행을 확인합니다.
mileage_failure_index = mileage_parts.index[mileage_failure_mask]
df_clean.loc[mileage_failure_index, ["name", "mileage"]]

,name,mileage


In [18]:
# engine의 숫자 부분과 단위 표기를 확인합니다.
engine_text = df_clean["engine"].dropna().astype(str).str.strip()

engine_parts = engine_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

engine_parts["unit"].value_counts(dropna=False)

unit
CC    6718
Name: count, dtype: int64

In [19]:
# engine의 숫자 부분 추출 실패 행 수를 확인합니다.
engine_failure_mask = engine_parts["number"].isna()
engine_failure_count = engine_failure_mask.sum()

engine_failure_count

np.int64(0)

In [20]:
# engine의 숫자 부분 추출 실패 행을 확인합니다.
engine_failure_index = engine_parts.index[engine_failure_mask]
df_clean.loc[engine_failure_index, ["name", "engine"]]

,name,engine


In [21]:
# max_power의 숫자 부분과 단위 표기를 확인합니다.
max_power_text = df_clean["max_power"].dropna().astype(str).str.strip()

max_power_parts = max_power_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

max_power_parts["unit"].value_counts(dropna=False)

unit
bhp    6717
          3
NaN       1
Name: count, dtype: int64

In [22]:
# max_power의 숫자 부분 추출 실패 행 수를 확인합니다.
max_power_failure_mask = max_power_parts["number"].isna()
max_power_failure_count = max_power_failure_mask.sum()

max_power_failure_count

np.int64(1)

In [23]:
# max_power의 숫자 부분 추출 실패 행을 모두 확인합니다.
max_power_failure_index = max_power_parts.index[max_power_failure_mask]
df_clean.loc[max_power_failure_index, ["name", "max_power"]]

,name,max_power
4933,Maruti Omni CNG,bhp


In [24]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행 수를 확인합니다.
max_power_empty_unit_mask = (
    max_power_parts["number"].notna()
    & max_power_parts["unit"].eq("")
)
max_power_empty_unit_count = max_power_empty_unit_mask.sum()

max_power_empty_unit_count

np.int64(3)

In [25]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행을 모두 확인합니다.
max_power_empty_unit_index = max_power_parts.index[max_power_empty_unit_mask]
df_clean.loc[max_power_empty_unit_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [26]:
# max_power에서 추출한 숫자 중 0 이하인 행을 확인합니다.
max_power_number = pd.to_numeric(max_power_parts["number"], errors="coerce")
max_power_non_positive_mask = max_power_number.le(0)
max_power_non_positive_index = max_power_number.index[max_power_non_positive_mask]

df_clean.loc[max_power_non_positive_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [27]:
# torque에서 사용되는 단위 유형을 확인합니다.
torque_text = df_clean["torque"].dropna().astype(str).str.strip()
torque_lower = torque_text.str.lower()

torque_nm_mask = torque_lower.str.contains("nm", regex=False)
torque_kgm_mask = torque_lower.str.contains("kgm", regex=False)
torque_other_mask = ~torque_nm_mask & ~torque_kgm_mask

pd.Series({
    "전체 non-null": len(torque_text),
    "nm 포함": torque_nm_mask.sum(),
    "kgm 포함": torque_kgm_mask.sum(),
    "nm과 kgm 미포함": torque_other_mask.sum(),
})

전체 non-null    6717
nm 포함          6227
kgm 포함          481
nm과 kgm 미포함      10
dtype: int64

In [28]:
# nm과 kgm 어느 것도 포함하지 않은 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_other_mask].value_counts()

torque
210 / 1900           7
250@ 1250-5000rpm    1
510@ 1600-2400       1
110(11.2)@ 4800      1
Name: count, dtype: int64

In [29]:
# torque의 주요 표기 방식별 행 수를 확인합니다.
torque_at_sign_mask = torque_text.str.contains("@", regex=False)
torque_at_word_mask = torque_lower.str.contains("at", regex=False)
torque_range_mask = torque_text.str.contains("-", regex=False)

pd.Series({
    "@ 포함": torque_at_sign_mask.sum(),
    "at 포함": torque_at_word_mask.sum(),
    "- 포함": torque_range_mask.sum(),
})

@ 포함     6493
at 포함     212
- 포함     2196
dtype: int64

In [30]:
# @ 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_sign_mask].drop_duplicates().head(10)

0            190Nm@ 2000rpm
1       250Nm@ 1500-2500rpm
2     12.7@ 2,700(kgm@ rpm)
4     11.5@ 4,500(kgm@ rpm)
5         113.75nm@ 4000rpm
6      7.8@ 4,500(kgm@ rpm)
7             59Nm@ 2500rpm
8       170Nm@ 1800-2400rpm
9            160Nm@ 2000rpm
10           248Nm@ 2250rpm
Name: torque, dtype: str

In [31]:
# at 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_word_mask].drop_duplicates().head(10)

3      22.4 kgm at 1750-2750rpm
109           96 Nm at 3000 rpm
149          250 Nm at 2750 rpm
190           146Nm at 4800 rpm
193        14.9 KGM at 3000 RPM
226       11.4 kgm at 4,000 rpm
286      180 Nm at 1440-1500rpm
472         135 Nm at 2500  rpm
474     24 KGM at 1900-2750 RPM
641     260 Nm at 1800-2200 rpm
Name: torque, dtype: str

In [32]:
# 회전수 범위처럼 보이는 - 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_range_mask].drop_duplicates().head(10)

1          250Nm@ 1500-2500rpm
3     22.4 kgm at 1750-2750rpm
8          170Nm@ 1800-2400rpm
15         115Nm@ 3500-3600rpm
19       219.7Nm@ 1500-2750rpm
39         320Nm@ 1700-2700rpm
41         250Nm@ 1750-2500rpm
47         343Nm@ 1400-3400rpm
48         200Nm@ 1400-3400rpm
49         200Nm@ 1250-4000rpm
Name: torque, dtype: str

In [33]:
# 조사 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [34]:
# 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [35]:
# nm과 kgm을 동시에 포함하는 torque 행 수를 확인합니다.
both_unit_mask = (
    torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)

both_unit_mask.sum()

np.int64(1)

In [36]:
# nm과 kgm을 동시에 포함하는 원본 행을 확인합니다.
both_unit_index = torque_lower.index[both_unit_mask]
df_clean.loc[both_unit_index, ["name", "year", "torque"]]

,name,year,torque
778,Ford Endeavour Hurricane Limited Edition,2013,380Nm(38.7kgm)@ 2500rpm


In [37]:
# torque 단위 조건이 서로 겹치지 않도록 네 그룹의 행 수를 확인합니다.
nm_only_mask = (
    torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)
kgm_only_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)
neither_unit_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)

exclusive_unit_counts = pd.Series({
    "nm만 포함": nm_only_mask.sum(),
    "kgm만 포함": kgm_only_mask.sum(),
    "nm과 kgm 모두 포함": both_unit_mask.sum(),
    "nm과 kgm 모두 미포함": neither_unit_mask.sum(),
})
exclusive_unit_counts.loc["네 그룹 합계"] = exclusive_unit_counts.sum()

exclusive_unit_counts

nm만 포함            6226
kgm만 포함            480
nm과 kgm 모두 포함        1
nm과 kgm 모두 미포함      10
네 그룹 합계           6717
dtype: int64

In [38]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [39]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [40]:
# 기존 at 조건과 정확한 공백 포함 at 조건의 행 수를 비교합니다.
torque_exact_at_mask = torque_lower.str.contains(" at ", regex=False)

pd.Series({
    "기존 \"at\" 포함": torque_at_word_mask.sum(),
    "정확한 \" at \" 포함": torque_exact_at_mask.sum(),
})

기존 "at" 포함       212
정확한 " at " 포함    212
dtype: int64

In [41]:
# 두 at 조건의 결과가 다른 행 수를 확인합니다.
torque_at_difference_mask = torque_at_word_mask != torque_exact_at_mask

torque_at_difference_mask.sum()

np.int64(0)

In [42]:
# 두 at 조건에서 차이가 발생하는 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_at_difference_mask].value_counts()

Series([], Name: count, dtype: int64)

In [43]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [44]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)